In [ ]:
#| eval: false
# --- Colab / fresh-environment setup -------------------------------------
# Running locally with pixi? You can skip this cell.
%pip install --quiet ratinabox


# Module 3 — Head Direction, Motion Signals & the Cell Zoo

Modules 1 and 2 gave us a moving animal and a code for where it is. This module covers the signals that say which way the animal is facing and how it is moving, plus a short tour of the other cell types RatInABox ships. It ends by assembling several populations into a single input matrix, which is the format any downstream computational model wants to work with.

> **Head Direction**: A head-direction cell fires when the animal faces a particular direction, roughly independent of where it is. They were first described in freely moving rats (Taube et al., 1990), and are found in anterior thalamus among other areas (Taube, 1995). The signal is best understood as an internally maintained heading estimate that is updated by self-motion (Taube, 2007). RatInABox models each cell as a bump of activity over heading angle, and the width of that bump is controlled by `angular_spread_degrees` (George et al., 2024).

## Outcomes from this Module

- Recover a heading **angle** from `Ag.head_direction`, which is a **unit vector**, with `np.arctan2`
- Pull `head_direction` `(T, 2)` and `rot_vel` `(T,)` out of `Ag.get_history_arrays()`
- Build a `HeadDirectionCells` population and set `n` and `angular_spread_degrees`
- Read `hd.preferred_angles` and draw polar maps with `plot_angular_rate_map()`
- Probe an **arbitrary** heading with `get_state(evaluate_at=None, head_direction=...)`
- **Measure** tuning width and show that a smaller `angular_spread_degrees` means sharper tuning
- Build an **empirical** tuning curve by binning `firingrate` over heading
- Instantiate `VelocityCells` and `SpeedCell`, and know why a fresh `Agent` breaks `SpeedCell`
- Use `rot_vel` as a ready made angular-velocity channel that needs no cell class at all
- Assemble a `(T, n_hd + n_pc + 1)` input matrix by updating everything in **lockstep**
- Survey `GridCells`, `BoundaryVectorCells`, and `ObjectVectorCells`
- Recognise an input channel that would make a result trivial

## Tutorial

### 0. RiaB Boilerplate

Same setup as before. The new imports are the cell classes this module uses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ratinabox
from ratinabox.Environment import Environment
from ratinabox.Agent import Agent
from ratinabox.Neurons import (
    PlaceCells,
    HeadDirectionCells,
    VelocityCells,
    SpeedCell,
    GridCells,
    BoundaryVectorCells,
    ObjectVectorCells,
)

ratinabox.autosave_plots = False
np.random.seed(0)               # for reproducibility
rng = np.random.default_rng(0)  # for sampling

### 1. How heading is represented

RiaB stores the current heading as a **unit vector**, not an angle. To get a scalar angle back, take `np.arctan2(y, x)`.

In [ ]:
env = Environment(params={"scale": 1.0})    # 1 m x 1 m open arena
Ag = Agent(env, params={"dt": 0.05})
Ag.update()                                 # one step, so the heading is defined

hdv = Ag.head_direction
print("head_direction ->", np.round(hdv, 3), hdv.shape)
print("its norm       ->", round(float(np.linalg.norm(hdv)), 6))
print("as an angle    ->", round(float(np.arctan2(hdv[1], hdv[0])), 3), "rad")

> **Gotcha:** `Ag.head_direction` is a **unit vector**, not an angle. It is shape `(2,)` and its length is 1. If you want a number in radians you have to call `np.arctan2` yourself, and the order of the arguments is `(y, x)`, not `(x, y)`.

The same holds in the history. `head_direction` comes back as `(T, 2)`, one unit vector per row, and there is a second motion channel sitting right next to it: `rot_vel`, the signed angular velocity in rad/s. Do the `arctan2` on the whole column at once rather than looping.

In [ ]:
for _ in range(int(30 / 0.05)):        # 30 s at dt = 0.05
    Ag.update()

h = Ag.get_history_arrays()
hd_vec = h["head_direction"]                           # (T, 2), each row a unit vector
heading = np.arctan2(hd_vec[:, 1], hd_vec[:, 0])       # (T,) radians in (-pi, pi]
rot_vel = h["rot_vel"]                                 # (T,) rad/s, signed

print("head_direction ->", hd_vec.shape)
print("heading        ->", heading.shape,
      f"range {heading.min():+.2f} to {heading.max():+.2f} rad")
print("rot_vel        ->", rot_vel.shape,
      f"range {rot_vel.min():+.2f} to {rot_vel.max():+.2f} rad/s")
print("all rows unit length:", bool(np.allclose(np.linalg.norm(hd_vec, axis=1), 1.0)))

Plotted against time, `heading` is the direction the animal faces and `rot_vel` is how fast that direction is changing. The vertical jumps in the top panel are the `arctan2` wrap at plus or minus pi, and they are an artefact of the angle, not of the motion. The unit vector has no such jumps.

In [ ]:
t = h["t"]
fig, axs = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)
axs[0].plot(t, heading, lw=0.8)
axs[0].set_ylabel("heading (rad)")
axs[0].axhline(np.pi, color="k", lw=0.5, ls=":")
axs[0].axhline(-np.pi, color="k", lw=0.5, ls=":")
axs[1].plot(t, rot_vel, lw=0.8, color="C3")
axs[1].axhline(0, color="k", lw=0.6)
axs[1].set_ylabel("rot_vel (rad/s)")
axs[1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

### 2. Head-direction cells

`HeadDirectionCells` takes the `Agent` as its first positional argument, exactly like `PlaceCells`. Two params matter: `n` (how many cells) and `angular_spread_degrees` (how wide each cell's tuning bump is). The cells sit at evenly spaced preferred headings around the circle, which you can read off `preferred_angles`.

In [ ]:
hd = HeadDirectionCells(Ag, params={"n": 10, "angular_spread_degrees": 45})

print("n                ->", hd.n)
print("preferred_angles ->", np.round(hd.preferred_angles, 3))
print("get_state()      ->", hd.get_state().shape, " (n, 1), NOT (n,)")
print("firing now       ->", np.round(hd.get_state()[:, 0], 3))

The shape convention is the one from module 2: `(n, n_positions)`, and a plain `get_state()` evaluates at the agent's current state so `n_positions` is 1. RiaB draws the polar tuning maps for you.

In [ ]:
hd.plot_angular_rate_map(chosen_neurons="3")     # first 3 cells, polar axes
plt.show()

To read a cell's tuning **without** moving the agent, hand `get_state` a heading of your own.

> **Gotcha:** to probe an arbitrary heading you must pass `evaluate_at=None` **together with** `head_direction=...`. A plain `get_state(head_direction=...)` keeps the default `evaluate_at="agent"`, silently uses the agent's own current heading, and ignores your `kwarg` entirely. This can result in silent failures, as not syntax error has occured.

In [ ]:
theta = 1.0                                            # a heading, in radians
probe = np.array([np.cos(theta), np.sin(theta)])       # as a unit vector

right = hd.get_state(evaluate_at=None, head_direction=probe)   # honours the probe
wrong = hd.get_state(head_direction=probe)                     # silently ignores it

print("with evaluate_at=None ->", np.round(right[:, 0], 3))
print("without               ->", np.round(wrong[:, 0], 3))
print()
print("the second one is just the agent's own heading:",
      bool(np.allclose(wrong, hd.get_state())))
print("peak of the probed response sits at cell",
      int(np.argmax(right[:, 0])),
      "whose preferred angle is",
      round(float(hd.preferred_angles[np.argmax(right[:, 0])]), 3), "rad")

### 3. Tuning width

`angular_spread_degrees` sets how broad the bump is. Here it is measured by using a sweep a probe heading all the way around the circle, then ask over what fraction of the circle the rate stays above half its peak. Do it for a sharp population and a broad one.

In [ ]:
angles = np.linspace(-np.pi, np.pi, 721)     # 0.5 degree steps


def tuning_curve(cells, cell=0):
    """Analytic response of one cell as a probe heading sweeps the circle."""
    return np.array([
        cells.get_state(evaluate_at=None,
                        head_direction=np.array([np.cos(a), np.sin(a)]))[cell, 0]
        for a in angles
    ])


def width_above_half(curve):
    """Angular width in degrees over which the rate exceeds half the peak."""
    return float((curve > 0.5 * curve.max()).mean() * 360.0)


sharp = HeadDirectionCells(Ag, params={"n": 1, "angular_spread_degrees": 30})
broad = HeadDirectionCells(Ag, params={"n": 1, "angular_spread_degrees": 90})

c_sharp = tuning_curve(sharp)
c_broad = tuning_curve(broad)

for name, spread, c in (("sharp", 30, c_sharp), ("broad", 90, c_broad)):
    print(f"{name:5s}  angular_spread_degrees={spread:3d}  "
          f"peak={c.max():.3f}  width above half max={width_above_half(c):.1f} deg")

About 71 degrees against about 270 degrees. Read the direction of that carefully, because the name invites the wrong guess: **a smaller `angular_spread_degrees` gives sharper tuning**. At 90 the bump covers three quarters of the circle, so the cell is barely selective at all. Both peak at 1.0, because `max_fr` defaults to 1 and the spread only changes the shape.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(np.degrees(angles), c_sharp, label="angular_spread_degrees = 30")
ax.plot(np.degrees(angles), c_broad, label="angular_spread_degrees = 90")
ax.axhline(0.5, color="k", lw=0.6, ls=":")
ax.set_xlabel("probe heading (degrees)")
ax.set_ylabel("firing rate")
ax.set_xlim(-180, 180)
ax.legend()
plt.tight_layout()
plt.show()

### 4. Tuning curves from a real run

The sweep above is the model's own tuning curve, read straight out of the equations. The other way to get a tuning curve is the way you would get one from a recording: run the animal, record the rate, and bin the rate by heading. Both should give the same shape.

In [ ]:
env4 = Environment(params={"scale": 1.0})
Ag4 = Agent(env4, params={"dt": 0.05})
hd4 = HeadDirectionCells(Ag4, params={"n": 4, "angular_spread_degrees": 45})

for _ in range(int(200 / 0.05)):       # 200 s, long enough to face every direction
    Ag4.update()
    hd4.update()

h4 = Ag4.get_history_arrays()
theta4 = np.arctan2(h4["head_direction"][:, 1], h4["head_direction"][:, 0])   # (T,)
fr4 = hd4.get_history_arrays()["firingrate"]                                  # (T, 4)
print("heading   ->", theta4.shape)
print("firingrate->", fr4.shape)

n_bins = 24
edges = np.linspace(-np.pi, np.pi, n_bins + 1)
centres = 0.5 * (edges[:-1] + edges[1:])
which = np.clip(np.digitize(theta4, edges) - 1, 0, n_bins - 1)
counts = np.bincount(which, minlength=n_bins)

cell = 0
empirical = np.array([fr4[which == b, cell].mean() if counts[b] else np.nan
                      for b in range(n_bins)])

print("samples per heading bin: min", counts.min(), "max", counts.max())
print("peak bin at", f"{np.degrees(centres[np.nanargmax(empirical)]):+.0f} deg")

In [ ]:
fig = plt.figure(figsize=(4.5, 4.5))
ax = fig.add_subplot(111, projection="polar")
ang = np.append(centres, centres[0])
val = np.append(empirical, empirical[0])
ax.plot(ang, val, "o-")
ax.fill(ang, val, alpha=0.2)
ax.set_title(f"empirical tuning, cell {cell}", pad=18)
plt.tight_layout()
plt.show()

One clear bump. The curve is a little lumpy?? compared to the first analytic sweep. The "lumpiness" is probably uneven sampling of headings. The animal does not spend equal time facing every direction in a finite run, so some bins are built from fewer samples than others. Run longer and the wobble shrinks. You can overlay the two curves so you can see how close they get.

### 5. Velocity and speed signals

RiaB packages self-motion as neurons too. `VelocityCells` codes the velocity **vector** and defaults to `n=10`. `SpeedCell` codes scalar speed and is a single cell.

In [ ]:
env5 = Environment(params={"scale": 1.0})
Ag5 = Agent(env5, params={"dt": 0.05})
sc = SpeedCell(Ag5)

try:
    sc.get_state()
except IndexError as e:
    print("on a fresh Agent  -> IndexError:", e)

Ag5.update()                       # one step is all it takes
vc = VelocityCells(Ag5)

print("after one update  ->", sc.get_state().shape, np.round(sc.get_state(), 3))
print("SpeedCell     n =", sc.n, " get_state() ->", sc.get_state().shape)
print("VelocityCells n =", vc.n, " get_state() ->", vc.get_state().shape)

> **Gotcha:** two things about `SpeedCell`. First, the name is **singular**. There is no `SpeedCells`. Second, it reads the last velocity out of `Agent.history["vel"]`, so on a brand new `Agent` that history is empty and `get_state()` raises `IndexError: list index out of range`. Call `Ag.update()` at least once before you read it. Sort of priming it...

There is also a motion channel that needs no cell class at all. `rot_vel` from the agent history is already a signed angular velocity in rad/s, one scalar per timestep. If what you want is an angular-velocity input, I think we can take it straight from the history and skip the neurons.

In [ ]:
ang_vel = rot_vel[:, None]          # (T,) -> (T, 1), ready to concatenate as a column
print("rot_vel      ->", rot_vel.shape)
print("as a column  ->", ang_vel.shape)
print(f"mean {rot_vel.mean():+.3f} rad/s, std {rot_vel.std():.3f} rad/s")
print("signed, so both turn directions are kept:",
      f"{(rot_vel > 0).mean():.2f} of steps positive")

### 6. Assembling a multi channel input matrix

Most downstream work wants one array: rows are timesteps, columns are channels. Build the populations you want on one agent, step all of them together, and concatenate the `firingrate` blocks along the feature axis. Scalar channels like `rot_vel` join as one more column.

In [ ]:
env6 = Environment(params={"scale": 1.0})
Ag6 = Agent(env6, params={"dt": 0.05})

n_hd, n_pc = 10, 8
hd6 = HeadDirectionCells(Ag6, params={"n": n_hd, "angular_spread_degrees": 45})
pc6 = PlaceCells(Ag6, params={"n": n_pc, "description": "gaussian", "widths": 0.2})

for _ in range(400):                # agent and BOTH populations, every single step
    Ag6.update()
    hd6.update()
    pc6.update()

h6 = Ag6.get_history_arrays()
X_hd = hd6.get_history_arrays()["firingrate"]      # (T, n_hd)
X_pc = pc6.get_history_arrays()["firingrate"]      # (T, n_pc)
X_av = h6["rot_vel"][:, None]                      # (T, 1)

X = np.concatenate([X_hd, X_pc, X_av], axis=1)

print("head-direction rates :", X_hd.shape)
print("place rates          :", X_pc.shape)
print("angular velocity     :", X_av.shape)
print("X                    :", X.shape, f"= (T, {n_hd} + {n_pc} + 1)")
print("all finite           :", bool(np.isfinite(X).all()))

> **Gotcha:** those three blocks only line up because the agent and the cells were stepped **in lockstep**, inside one loop, from the moment the cells were created. Step the agent even once before the populations exist and its history is one row longer than theirs forever after. `np.concatenate` then fails with
>
> ```
> all the input array dimensions except for the concatenation axis must match exactly,
> but along dimension 0, the array at index 0 has size 400 and the array at index 2 has size 401
> ```
>
> I find this is the error you will often hit with this library, and the fix has been: create every population first, then run one loop that updates all of them.

Here is that failure on purpose, so you recognise it when it turns up for real.

In [ ]:
env_bad = Environment(params={"scale": 1.0})
Ag_bad = Agent(env_bad, params={"dt": 0.05})
Ag_bad.update()                              # <-- one step BEFORE the cells exist
hd_bad = HeadDirectionCells(Ag_bad, params={"n": n_hd, "angular_spread_degrees": 45})
pc_bad = PlaceCells(Ag_bad, params={"n": n_pc, "description": "gaussian",
                                    "widths": 0.2})

for _ in range(400):                         # the same lockstep loop as above
    Ag_bad.update()
    hd_bad.update()
    pc_bad.update()

h_bad = Ag_bad.get_history_arrays()
fr_hd_bad = hd_bad.get_history_arrays()["firingrate"]
fr_pc_bad = pc_bad.get_history_arrays()["firingrate"]
print("cell  history rows:", fr_hd_bad.shape[0])
print("agent history rows:", h_bad["rot_vel"].shape[0], "  <-- one too many")

try:
    np.concatenate([fr_hd_bad, fr_pc_bad, h_bad["rot_vel"][:, None]], axis=1)
except ValueError as e:
    print("\nValueError:", e)

### 7. The wider cell zoo

Place cells and head-direction cells are the two you will reach for most, but RiaB ships several more. Let's look at a few...

**Grid cells** fire on a repeating, I believe triangular, lattice that tiles the whole environment, and were described in medial entorhinal cortex (Hafting et al., 2005). `GridCells` gives you `gridscale`, `orientation`, and `phase_offset` per cell, plus distribution params to sample them.

In [ ]:
gc = GridCells(Ag, params={"n": 9})
print("get_state()               ->", gc.get_state().shape)
print("get_state(evaluate_at=all)->", gc.get_state(evaluate_at="all").shape)
print("params:", sorted(GridCells.default_params.keys()))

gc.plot_rate_map(chosen_neurons="3")
plt.show()

**Boundary vector cells** fire when a boundary sits at a particular distance and a particular direction from the animal, and were recorded in the hippocampal formation (Lever et al., 2009). The rate map is a stripe running parallel to whichever wall the cell prefers.

In [ ]:
bvc = BoundaryVectorCells(Ag, params={"n": 8})
print("get_state() ->", bvc.get_state().shape)
print("params:", sorted(BoundaryVectorCells.default_params.keys()))

bvc.plot_rate_map(chosen_neurons="3")
plt.show()

**Object vector cells** are the same idea measured against an object rather than a wall, and were described in medial entorhinal cortex (Hoydal et al., 2019). RiaB needs an object to exist before you can build them, so call `env.add_object([x, y])` on the environment first.

In [ ]:
env_obj = Environment(params={"scale": 1.0})
env_obj.add_object([0.5, 0.5])                 # must come BEFORE the cells
Ag_obj = Agent(env_obj, params={"dt": 0.05})

ovc = ObjectVectorCells(Ag_obj, params={"n": 4})
print("get_state() ->", ovc.get_state().shape)
print("params:", sorted(ObjectVectorCells.default_params.keys()))

# RiaB colours OVCs by their preferred angle, which is an RGBA array and trips up
# plot_rate_map. Clearing it lets the plotter fall back to its default colour.
ovc.color = None
ovc.plot_rate_map(chosen_neurons="2")
plt.show()

### 8. A note on what not to feed a model

One design rule is worth stating before you build a dataset out of any of this. If you are testing whether some structure emerges in a trained model, do not hand the model a channel that is a deterministic function of that structure. If you do, the result can be read straight off the input rather than learned, and the experiment answers nothing.

The concrete example is problem 3.5. A channel defined as heading modulo pi is one line of numpy and it looks harmless. It is not. It throws away the difference between a heading and the heading 180 degrees away, so any structure that is invariant to that flip is handed over for free. As one example of why that matters, some subiculum neurons fire both for a given heading and for the heading 180 degrees opposite (Olson et al., 2017). If you wanted to ask whether a trained model develops that kind of tuning on its own, handing it heading modulo pi would answer the question before training began. 

For every column you plan to feed in, ask what function of the trajectory it is, and whether the thing you are looking for can be recovered from it directly. Raw kinematic channels like `rot_vel` and population rates from cells defined on position or heading are usually fine. Channels you construct yourself out of the very quantity you are testing usually are not.

---

## Key API

| Task | Call |
|---|---|
| Heading now | `Ag.head_direction` → a **unit vector**, shape `(2,)` |
| Heading as an angle | `np.arctan2(hd[1], hd[0])` (note the `(y, x)` order) |
| Heading over a run | `Ag.get_history_arrays()["head_direction"]` → `(T, 2)` |
| Angular velocity | `Ag.get_history_arrays()["rot_vel"]` → `(T,)` signed, rad/s |
| Heading angles over a run | `np.arctan2(hd[:,1], hd[:,0])` → `(T,)` in `(-pi, pi]` |
| HD population | `HeadDirectionCells(Ag, params={"n":10, "angular_spread_degrees":45})` |
| Tuning width | `angular_spread_degrees` (**smaller means sharper**) |
| Preferred headings | `hd.preferred_angles` → `(n,)` evenly spaced radians |
| Firing now | `hd.get_state()` → `(n, 1)`, squeeze with `[:, 0]` |
| Firing at a chosen heading | `hd.get_state(evaluate_at=None, head_direction=np.array([np.cos(a), np.sin(a)]))` |
| Polar tuning maps | `hd.plot_angular_rate_map(chosen_neurons="3")` |
| Heading-averaged state | `hd.get_head_direction_averaged_state()` |
| Velocity cells | `VelocityCells(Ag)` → `n=10`, `get_state()` → `(n, 1)` |
| Speed cell | `SpeedCell(Ag)` (**singular**) → `n=1`, `get_state()` → `(1,)` |
| Step everything | one loop: `Ag.update()`, then `hd.update()`, then `pc.update()` |
| Rate matrix | `cells.get_history_arrays()["firingrate"]` → `(T, n)` |
| Scalar as a column | `rot_vel[:, None]` → `(T, 1)` |
| Stack channels | `np.concatenate([X_hd, X_pc, X_av], axis=1)` → `(T, n_hd + n_pc + 1)` |
| Grid cells | `GridCells(Ag, params={"n":9})`, params include `gridscale`, `orientation`, `phase_offset` |
| Boundary vector cells | `BoundaryVectorCells(Ag, params={"n":8})`, params include `dtheta` |
| Object vector cells | `env.add_object([0.5, 0.5])` **first**, then `ObjectVectorCells(Ag, params={"n":4})` |

---

## Problems

Answers are folded below each problem. Try it yourself first.

### Problem 3.1

Build two head-direction populations that differ only in `angular_spread_degrees`, one at 30 and one at 90. Plot polar angular rate maps for a few cells from each, and describe what tuning width does.

In [ ]:
# TODO: your answer here.
# Hint: the module page for this notebook walks through the API you need.


### Problem 3.2

Assemble a multi channel input matrix for a short run: head-direction rates `(T, n_hd)`, place rates `(T, n_pc)`, and a scalar angular-velocity channel taken from `rot_vel`. Concatenate them into one matrix `X` of shape `(T, n_hd + n_pc + 1)`.

**Hint:** update the agent and every cell population in lockstep inside one loop, otherwise the histories end up different lengths and the concatenate fails. `rot_vel` is 1-D, so add an axis with `[:, None]`.

In [ ]:
# TODO: your answer here.
# Hint: the module page for this notebook walks through the API you need.


### Problem 3.3

Compute a head-direction tuning curve empirically by binning firing rate over heading angle. Overlay the analytic curve obtained by probing headings directly, and confirm the cell is unimodal.

**Hint:** get the analytic curve with `get_state(evaluate_at=None, head_direction=...)` at each probe angle. Normalise both curves before overlaying so the shapes are comparable.

In [ ]:
# TODO: your answer here.
# Hint: the module page for this notebook walks through the API you need.


### Problem 3.4

Instantiate `GridCells` and `BoundaryVectorCells`, plot a few rate maps of each, and write two or three sentences on what each one represents and where it is found in the brain.

In [ ]:
# TODO: your answer here.
# Hint: the module page for this notebook walks through the API you need.


### Problem 3.5

Build a channel from the trajectory defined as heading modulo pi. Show numerically that two opposite headings map to the identical value, and explain in prose why a channel like this must be kept out of a model's inputs when the thing you are testing for is direction related structure. Tuning that is bimodal in exactly this way has been reported in subiculum (Olson et al., 2017), so this is not a hypothetical concern.

**Hint:** `np.mod(angle, np.pi)`. Compare `np.mod(0.3, np.pi)` with `np.mod(0.3 + np.pi, np.pi)`. Then check the correlation between this channel and the raw heading to make the point quantitative.

In [ ]:
# TODO: your answer here.
# Hint: the module page for this notebook walks through the API you need.


# References

@article{george2024ratinabox,
  author  = {George, Tom M. and Rastogi, Mehul and de Cothi, William and
             Clopath, Claudia and Stachenfeld, Kimberly and Barry, Caswell},
  title   = {{RatInABox}, a toolkit for modelling locomotion and neuronal
             activity in continuous environments},
  journal = {eLife},
  volume  = {13},
  pages   = {e85274},
  year    = {2024},
  doi     = {10.7554/eLife.85274}
}

@article{taube1990headdirection,
  author  = {Taube, Jeffrey S. and Muller, Robert U. and Ranck, James B.},
  title   = {Head-direction cells recorded from the postsubiculum in freely
             moving rats. I. Description and quantitative analysis},
  journal = {The Journal of Neuroscience},
  volume  = {10},
  number  = {2},
  pages   = {420--435},
  year    = {1990},
  doi     = {10.1523/JNEUROSCI.10-02-00420.1990}
}

@article{taube1995anterior,
  author  = {Taube, Jeffrey S.},
  title   = {Head direction cells recorded in the anterior thalamic nuclei of
             freely moving rats},
  journal = {The Journal of Neuroscience},
  volume  = {15},
  number  = {1},
  pages   = {70--86},
  year    = {1995},
  doi     = {10.1523/JNEUROSCI.15-01-00070.1995}
}

@article{taube2007headdirection,
  author  = {Taube, Jeffrey S.},
  title   = {The head direction signal: origins and sensory-motor integration},
  journal = {Annual Review of Neuroscience},
  volume  = {30},
  pages   = {181--207},
  year    = {2007},
  doi     = {10.1146/annurev.neuro.29.051605.112854}
}

@article{hafting2005microstructure,
  author  = {Hafting, Torkel and Fyhn, Marianne and Molden, Sturla and
             Moser, May-Britt and Moser, Edvard I.},
  title   = {Microstructure of a spatial map in the entorhinal cortex},
  journal = {Nature},
  volume  = {436},
  number  = {7052},
  pages   = {801--806},
  year    = {2005},
  doi     = {10.1038/nature03721}
}

@article{lever2009boundary,
  author  = {Lever, Colin and Burton, Stephen and Jeewajee, Ali and
             O'Keefe, John and Burgess, Neil},
  title   = {Boundary vector cells in the subiculum of the hippocampal
             formation},
  journal = {The Journal of Neuroscience},
  volume  = {29},
  number  = {31},
  pages   = {9771--9777},
  year    = {2009},
  doi     = {10.1523/JNEUROSCI.1319-09.2009}
}

@article{hoydal2019object,
  author  = {H{\o}ydal, {\O}yvind Arne and Skyt{\o}en, Emilie Ranheim and
             Andersson, Sebastian O. and Moser, May-Britt and
             Moser, Edvard I.},
  title   = {Object-vector coding in the medial entorhinal cortex},
  journal = {Nature},
  volume  = {568},
  number  = {7752},
  pages   = {400--404},
  year    = {2019},
  doi     = {10.1038/s41586-019-1077-7}
}

@article{olson2017subiculum,
  author  = {Olson, Jake M. and Tongprasearth, Kanyanat and Nitz, Douglas A.},
  title   = {Subiculum neurons map the current axis of travel},
  journal = {Nature Neuroscience},
  volume  = {20},
  number  = {2},
  pages   = {170--172},
  year    = {2017},
  doi     = {10.1038/nn.4464}
}
